# **varlap to rust WIP**

## **Project Outline**

**Milestone 1**

- Implement a simple read first algorithm to make sure the concept works

**Milestone 2**

- Implement multithreading

**Milestone 3**

- Optimize program/make sure there’s even loading for each thread

**Milestone 4**

- Consider how to deal with overlapping in paired end sequencing reads
- Consider other statistics that may be useful with the information we have
- Maybe a feature that can save where you have progressed to if there is an error?

NOTES:

- Compare reads aligned in Bam in program to IGV (Note: default settings in IGV may not get all reads)
- Check how samtools/htslib gets the reads in a given region (default settings may skip low quality reads, or there may be a limit on how many reads are returned)

### **Setup notes**

To setup rust with jupyter:
- https://ratulmaharaj.com/posts/interactive-rust-with-jupyter-notebooks/

To setup varlap on Ubuntu with global Python 3.12:
- Error: `ERROR: Failed to build 'pysam' when getting requirements to build wheel`

```
$ git clone https://github.com/bjpop/varlap
$ cd varlap
$ sudo apt update
$ sudo apt install python3.7 python3.7-venv python3.7-distutils
$ python3.7 -m venv varlap_dev
$ source varlap_dev/bin/activate
$ pip install -U /path/to/varlap
```

Software used so far:

- WSL version: 2.6.3.0
- Kernel version: 6.6.87.2-1
- WSLg version: 1.0.71
- MSRDC version: 1.2.6353
- Direct3D version: 1.611.1-81528511
- DXCore version: 10.0.26100.1-240331-1435.ge-release
- Windows version: 10.0.26200.8037
- mamba version 2.2.3
- mamba install -c bioconda igv -y
- Ubuntu 24.04.3 LTS

### **Creating the structure for each variant:**
- In the original varlap, where each variant is looped over, variants are in a generator object (yield) to save memory
- However for our read first approach, we need to create objects/structs for all variants at the start and store all in the memory (Since we can’t be certain that for a given region, the first read in the bam file does not span the entire region)
- Therefore need to minimize what info we store in each variant structure
- Varlap stores: {"chrom": chrom, "pos": pos, "ref": ref, "alt": alt}, counts of bases, variant type, and associated statistics
- Skip statistics for now

#### Variant

Initialise it with what we know for chrom/pos/ref/alt (Note: ref can't be used so we use refr instead; maybe change to something else?)

In [2]:
#[derive(Debug, Clone)]
struct VariantReader {
	chrom: String,
	pos: u64,
	refr: String,
	alt: String,
}

#### Base Counts

For base counts we can use a struct; we also implement a counter for the structure so that every time we match increment the base count

In [3]:
#[derive(Debug, Clone, Default)]
struct BaseCounts {
    a: u64,
    c: u64,
    g: u64,
    t: u64,
    n: u64,
}


impl BaseCounts {
    fn increment(&mut self, base: char) {
        match base {
            'A' => self.a += 1,
            'C' => self.c += 1,
            'G' => self.g += 1,
            'T' => self.t += 1,
            'N' => self.n += 1,
            _ => eprintln!("Warning: Base does not match: {}", base),
        }
    }
}

#### Variant Types

The python code:
```
def get_var_type(ref, alt):
    if len(ref) == 1 and len(alt) == 1:
        return "SNV"
    elif len(ref) > len(alt):
        return "DEL"
    elif len(alt) > len(ref):
        return "INS"
    else:
        logging.warning(f"Cannot determine the type of variant with ref: {ref} and alt: {alt}")
        return "UNKNOWN"
```

For variant types we can use an enum to define each type (use enum instead of struct as types are mutually exclusive) instead of storing each type as a string like the python code

We also implement a function that converts the type into a string for later; use `&'static str` as we know they are fixed size literals; avoids allocating new string to each variant in heap

In [4]:
#[derive(Debug, Clone, Copy)]
enum VarType {
    Snv,
    Del,
    Ins,
    Unknown,
}

impl VarType {
    fn as_str(&self) -> &'static str {
        match self {
            VarType::Snv => "SNV",
            VarType::Del => "DEL",
            VarType::Ins => "INS",
            VarType::Unknown => "UNKNOWN",            
        }
    }
}



We also need to implement a function that gets the variant type from the ref and alt alleles of a variant

In [5]:
fn get_var_type(refr: &str, alt: & str) -> VarType {
    if refr.len() == 1 && alt.len() == 1 {
        VarType::Snv
    } else if refr.len() > alt.len() {
        VarType::Del
    } else if refr.len() < alt.len() {
        VarType::Ins
    } else {
        eprintln!(
            "Warning: Cannot determine the type of variant with ref: {} and alt: {}",
            refr, alt
        );
        VarType::Unknown
    }
}

Update the variant structure with what we have just implemented:

In [6]:
#[derive(Debug, Clone)]
struct Variant {
	chrom: String,
	pos: u64,
	refr: String,
	alt: String,
	vartype: VarType,
	counts: BaseCounts,
}

### **VCF Reader**
Now we need to implement a function that reads a VCF (implement csv/tsv later) and creates an vector of Variant structures for each variant
- Variants need to be in a queue as we want to drop them from memory once the start of the read position is > than the variant position

Python code:
```
def vcf_reader(file):
    for line in file:
        if line.startswith('#'):
            continue
        fields = line.strip().split()
        # Technically VCF requires the first 8 fields to be defined, but we want to be as liberal
        # as possible in accepting inputs.
        if len(fields) >= 5:
            chrom, pos, _id, ref, alt = fields[:5]
            yield {"chrom": chrom, "pos": pos, "ref": ref, "alt": alt}
        else:
            logging.warning(f"Skipping input row: {line}")
```
and
```
    def get_variants(self):
        '''Read variants from input VCF file, yield one at a time'''
        for input_row in self.reader: 
            self.total_variants_in_input += 1
            if is_valid_input_row(input_row):
                this_ref = input_row["ref"]
                # allow possibly multiple alts in the same variant, split them into separate alleles
                alts = input_row["alt"].split(",")
                for this_alt in alts:
                    this_var_type = get_var_type(this_ref, this_alt)
                    if is_acceptable_variant(dict(input_row), self.varclass, this_var_type, this_ref, this_alt, self.max_indel_size):
                        output_row = copy(input_row)
                        output_row["alt"] = this_alt
                        output_row["pos"] = int(input_row["pos"])
                        output_row["vartype"] = this_var_type
                        self.num_variants_analysed += 1
                        yield output_row
            else:
                logging.warning(f"Skipping invalid input row: {dict(input_row)}")
```

In [7]:
use std::fs::File;
use std::io::{self, BufRead, BufReader};
use std::collections::VecDeque;
use std::error::Error;

fn vcf_reader(file_path: &str) -> Result<VecDeque<Variant>, Box<dyn Error>> {
    let file = File::open(file_path)?;
    let reader = BufReader::new(file);

	let mut variants = VecDeque::new();

    for line_result in reader.lines() {
        let line = line_result?;
        if line.starts_with("#") {
            continue;
            }
        
        let fields: Vec<&str> = line.split_whitespace().collect();
        
        if fields.len() >= 5 {
            let chrom = fields[0].to_string();

            let pos = match fields[1].parse::<u64>() {
                Ok(p) => p,
                Err(_) => {
                    eprintln!("Warning: invalid POS, skipping row: {}", line);
                    continue;
                }
            };

            let refr = fields[3].to_string();

            for alt in fields[4].split(',') {
                let vartype = get_var_type(&refr, &alt);

                variants.push_back(Variant {
                    chrom: chrom.clone(),
                    pos,
                    refr: refr.clone(),
                    alt: alt.to_string(),
                    vartype,
                    counts: BaseCounts::default(),
                });
            }
        } else {
            eprintln!("Warning: Skipping input row: {}", line);
        }
    }
			
	Ok(variants)
}

Notes:

- Need `let line = line_result?;` as line yields `Option<Result<String, std::io::Error>>`; `?` unwraps the `Ok(String)` case or returns an error if it cannot read the line
- We use `.to_string()` to convert the string slice in `fields` to an actual `String` with ownership; we then have to use `.clone()` when looping over all alts, otherwise the first chrom/refr String will get consumed in the first iteration
- `u64` type has the `Copy` trait and does not get consumed so we do not need to clone; using the match field converts to an actual `u64` and skips if there is an error
- We multiple alts in the same variant by separating them and creating different Variant structs; we assume that they are separated by `,`
- **Should we use `Arc<str>` for chrom globally and for the reference allele locally for each variant? How often are there multiallelic alts for this to be worth it? Or store chrom as an id integer?**
- 

We also need to create a function that gets the min and max positions of all the variants (i.e. the interval that all the variants span); we can then use this to get all reads in the bam file that overlap this region/interval
- If we assume that the vcf contains variants from only one chromosome, and that they are sorted we can use:

In [12]:
fn get_vcf_min_max(variants: &VecDeque<Variant>) -> Option<(String, u64, u64)> {
    let first = variants.front()?;
    let chrom = first.chrom.clone();

    let min_pos = first.pos;
    let max_pos = variants.back()?.pos;

    Some((chrom, min_pos, max_pos))
}

#### Lets test our functions so far using a small vcf

In [24]:
use std::fs;

let file_path = "test_data/vars.vcf";

let data = fs::read_to_string(&file_path).expect("Should be able to read file");
println!("{}", data);

##fileformat=VCFv4.2
##contig=<ID=chr1,length=260>
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO
chr1	100	.	AT	A	.	.	.
chr1	115	.	G	C	.	.	.
chr1	130	.	C	T,G	.	.	.
chr1	145	.	A	G	.	.	.
chr1	160	.	T	C	.	.	.
chr1	175	.	G	A	.	.	.
chr1	190	.	C	CT	.	.	.
chr1	200	.	T	G	.	.	.



In [30]:
let variants = vcf_reader(&file_path)?;

let (region_chrom, min_pos, max_pos) = 
    get_vcf_min_max(&variants).ok_or("Could not determine VCF min/max")?;

println!("Region Chromosome: {}, Min Pos: {}, Max Pos: {} \n", region_chrom, min_pos, max_pos);

println!("All Variants in the Variants VecDeque:");
for line in &variants {
    println!("{:?}", line);
}

Region Chromosome: chr1, Min Pos: 100, Max Pos: 200 

All Variants in the Variants VecDeque:
Variant { chrom: "chr1", pos: 100, refr: "AT", alt: "A", vartype: Del, counts: BaseCounts { a: 0, c: 0, g: 0, t: 0, n: 0 } }
Variant { chrom: "chr1", pos: 115, refr: "G", alt: "C", vartype: Snv, counts: BaseCounts { a: 0, c: 0, g: 0, t: 0, n: 0 } }
Variant { chrom: "chr1", pos: 130, refr: "C", alt: "T", vartype: Snv, counts: BaseCounts { a: 0, c: 0, g: 0, t: 0, n: 0 } }
Variant { chrom: "chr1", pos: 130, refr: "C", alt: "G", vartype: Snv, counts: BaseCounts { a: 0, c: 0, g: 0, t: 0, n: 0 } }
Variant { chrom: "chr1", pos: 145, refr: "A", alt: "G", vartype: Snv, counts: BaseCounts { a: 0, c: 0, g: 0, t: 0, n: 0 } }
Variant { chrom: "chr1", pos: 160, refr: "T", alt: "C", vartype: Snv, counts: BaseCounts { a: 0, c: 0, g: 0, t: 0, n: 0 } }
Variant { chrom: "chr1", pos: 175, refr: "G", alt: "A", vartype: Snv, counts: BaseCounts { a: 0, c: 0, g: 0, t: 0, n: 0 } }
Variant { chrom: "chr1", pos: 190, ref

()